In [5]:
import pandas as pd 
import requests 
from bs4 import BeautifulSoup

**WEB SCRAPING**

In [3]:
import requests
from bs4 import BeautifulSoup
import re
import time

data = []

for start in range(0,2000,50):

    url = f'https://store.steampowered.com/search/results/?query&start={start}&count=50'

    webpage = requests.get(url)

    soup = BeautifulSoup(webpage.text,'lxml')

    cards = soup.find_all('a', class_='search_result_row')

    for card in cards:

     
        name = card.find('span', class_='title').text.strip()

    
        release_tag = card.find('div', class_='search_released')
        release_date = release_tag.text.strip() if release_tag else None

     
        price_tag = card.select_one('.discount_final_price, .search_price')
        price = price_tag.text.strip() if price_tag else None
        link = card['href']

       s
        review_tag = card.find('span', class_='search_review_summary')

        if review_tag:

            tooltip = review_tag.get('data-tooltip-html','')

            review_count = re.findall(r'\d[\d,]*', tooltip)
            review_count = review_count[0] if review_count else None

            rating_label = BeautifulSoup(tooltip,'lxml').text.split("\n")[0]

        else:
            review_count = None
            rating_label = None


    
        game_page = requests.get(link)

        game_soup = BeautifulSoup(game_page.text,"lxml")

     
        developer_tag = game_soup.find("div", id="developers_list")
        developer = developer_tag.text.strip() if developer_tag else None


        publisher = None
        rows = game_soup.find_all("div", class_="dev_row")

        for row in rows:
            subtitle = row.find("div", class_="subtitle column")

            if subtitle and "Publisher" in subtitle.text:
                publisher = row.find("a").text.strip()
                break


        
        genre_tags = game_soup.select(".details_block a")
        genres = ", ".join([g.text for g in genre_tags]) if genre_tags else None


        data.append({
            'game_name': name,
            'release_date': release_date,
            'price': price,
            'rating_label': rating_label,
            'review_count': review_count,
            'developer': developer,
            'publisher': publisher,
            'genres': genres
        })

        
        time.sleep(1)

    print("scraped_offset:", start)


scraped_offset: 0
scraped_offset: 50
scraped_offset: 100
scraped_offset: 150
scraped_offset: 200
scraped_offset: 250
scraped_offset: 300
scraped_offset: 350
scraped_offset: 400
scraped_offset: 450
scraped_offset: 500
scraped_offset: 550
scraped_offset: 600
scraped_offset: 650
scraped_offset: 700
scraped_offset: 750
scraped_offset: 800
scraped_offset: 850
scraped_offset: 900
scraped_offset: 950
scraped_offset: 1000
scraped_offset: 1050
scraped_offset: 1100
scraped_offset: 1150
scraped_offset: 1200
scraped_offset: 1250
scraped_offset: 1300
scraped_offset: 1350
scraped_offset: 1400
scraped_offset: 1450
scraped_offset: 1500
scraped_offset: 1550
scraped_offset: 1600
scraped_offset: 1650
scraped_offset: 1700
scraped_offset: 1750
scraped_offset: 1800
scraped_offset: 1850
scraped_offset: 1900
scraped_offset: 1950


In [5]:
import pandas as pd

df = pd.DataFrame(data)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   game_name     2000 non-null   object
 1   release_date  2000 non-null   object
 2   price         1991 non-null   object
 3   rating_label  1983 non-null   object
 4   review_count  1983 non-null   object
 5   developer     1996 non-null   object
 6   publisher     1967 non-null   object
 7   genres        1997 non-null   object
dtypes: object(8)
memory usage: 125.1+ KB


In [13]:
df.head()


,game_name,release_date,price,rating_label,review_count,developer,publisher,genres
0,Slay the Spire 2,"5 Mar, 2026","₹1,100.00","Overwhelmingly Positive97% of the 13,496 user ...",97,Mega Crit,Mega Crit,"Indie, Strategy, Early Access, Mega Crit, Mega..."
1,Resident Evil Requiem,"26 Feb, 2026","₹4,399.00","Overwhelmingly Positive96% of the 34,291 user ...",96,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.","Action, Adventure, CAPCOM Co., Ltd., CAPCOM Co..."
2,Counter-Strike 2,"21 Aug, 2012",Free,"Very Positive86% of the 2,496,132 user reviews...",86,Valve,Valve,"Action, Free To Play, Valve, Valve, \n\t\t\tVi..."
3,Marathon,"5 Mar, 2026","₹2,449.00","Very Positive90% of the 12,119 user reviews fo...",90,Bungie,Bungie,"Action, Bungie, Bungie, \n\t\t\tVisit the webs..."
4,ARC Raiders,"30 Oct, 2025","₹2,467.00","Very Positive86% of the 175,346 user reviews f...",86,Embark Studios,Embark Studios,"Action, Embark Studios, Embark Studios, \n\t\t..."


In [11]:
df.to_csv('steam_games.csv')

**Data preprocessing**

In [7]:
df = pd.read_csv('steam_games.csv')

In [9]:
df['game_name'].duplicated().sum()

17

In [11]:
df[df['game_name'].duplicated(keep=False)]

,Unnamed: 0,game_name,release_date,price,rating_label,review_count,developer,publisher,genres
146,146,Age of Empires II: Definitive Edition,"14 Nov, 2019","₹1,399.00","Overwhelmingly Positive95% of the 59,282 user ...",95.0,"World's Edge, Forgotten Empires, CaptureAge, T...",Xbox Game Studios,"Strategy, World's Edge, Forgotten Empires, Cap..."
1199,1199,Just Cause™ 3,"30 Nov, 2015",₹399.00,"Very Positive86% of the 44,728 user reviews fo...",86.0,Avalanche Studios,Square Enix,"Action, Adventure, Avalanche Studios, Square E..."
1206,1206,Just Cause™ 3,"30 Nov, 2015",₹399.00,"Very Positive86% of the 44,728 user reviews fo...",86.0,Avalanche Studios,Square Enix,"Action, Adventure, Avalanche Studios, Square E..."
1233,1233,Scrap Mechanic,"19 Jan, 2016",₹565.00,"Very Positive90% of the 45,677 user reviews fo...",90.0,Axolot Games,Axolot Games,"Action, Adventure, Indie, Simulation, Early Ac..."
1239,1239,Need for Speed™ Most Wanted,"18 Jun, 2020",₹999.00,"Mostly Positive74% of the 7,949 user reviews f...",74.0,Criterion Games,Electronic Arts,"Action, Adventure, Racing, Criterion Games, El..."
1241,1241,A Game About Feeding A Black Hole,"15 Dec, 2025",₹345.00,"Very Positive89% of the 1,423 user reviews for...",89.0,"Aarimous, Thornityco",Aarimous,"Casual, Indie, Aarimous, Thornityco, Aarimous,..."
1242,1242,Forts,"19 Apr, 2017",₹479.00,"Very Positive93% of the 8,432 user reviews for...",93.0,EarthWork Games,EarthWork Games,"Action, Indie, Strategy, EarthWork Games, Eart..."
1249,1249,Marvel's Midnight Suns,"1 Dec, 2022",₹918.00,"Very Positive83% of the 11,238 user reviews fo...",83.0,Firaxis Games,2K,"RPG, Strategy, Firaxis Games, 2K, Marvel's Mid..."
1250,1250,Marvel's Midnight Suns,"1 Dec, 2022",₹918.00,"Very Positive83% of the 11,238 user reviews fo...",83.0,Firaxis Games,2K,"RPG, Strategy, Firaxis Games, 2K, Marvel's Mid..."
1252,1252,Scrap Mechanic,"19 Jan, 2016",₹565.00,"Very Positive90% of the 45,677 user reviews fo...",90.0,Axolot Games,Axolot Games,"Action, Adventure, Indie, Simulation, Early Ac..."


In [13]:
df = df.drop_duplicates(subset='game_name')

In [15]:
df['game_name'].duplicated().sum()

0

In [17]:
df['price'] = df['price'].str.replace('₹','')
df['price'] = df['price'].str.replace(',','')
df['price'] = df['price'].str.replace('Free','0')
df['price'] = df['price'].astype(float)

In [19]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month

df = df.dropna(subset=['release_year'])

df['release_year'] = df['release_year'].astype(int)
df['release_month'] = df['release_month'].astype(int)

df = df.drop(columns=['release_date'])

In [21]:
df['review_sentiment'] = df['rating_label'].str.extract(r'^(.*?)(?=\d)')

In [23]:
df['review_score'] = df['rating_label'].str.extract(r'(\d+)%')

df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce')

df = df.dropna(subset=['review_score'])

df['review_score'] = df['review_score'].astype(int)

In [25]:
df['review_count'] = df['rating_label'].str.extract(r'([\d,]+)\suser')
df['review_count'] = df['review_count'].str.replace(',', '')
df['review_count'] = df['review_count'].astype(int)

In [27]:
df.isnull().sum()

Unnamed: 0           0
game_name            0
price                6
rating_label         0
review_count         0
developer            1
publisher           30
genres               1
release_year         0
release_month        0
review_sentiment     0
review_score         0
dtype: int64

In [29]:
df.drop(columns = ['Unnamed: 0'] , inplace = True)

In [31]:
df.drop(columns = ['rating_label'] , inplace = True)

In [33]:
top_devs = df['developer'].value_counts().nlargest(20).index

df['developer'] = df['developer'].apply(lambda x: x if x in top_devs else 'Other')

In [35]:
top_pubs = df['publisher'].value_counts().nlargest(20).index

df['publisher'] = df['publisher'].apply(lambda x: x if x in top_pubs else 'Other')

In [37]:
df['genres'] = df['genres'].str.split(',').str[:3]
df['genres'] = df['genres'].apply(lambda x: ','.join(x) if isinstance(x,list) else x)

In [39]:
valid_genres = [
'Action','Adventure','RPG','Strategy','Simulation','Sports',
'Racing','Casual','Indie','Free To Play','Massively Multiplayer',
'Early Access','Puzzle','Horror','Platformer','Shooter'
]

def clean_genres(genres):
    
    if pd.isna(genres):
        return None
        
    items = [g.strip() for g in genres.split(',')]
    
    filtered = [g for g in items if g in valid_genres]
    
    return ','.join(filtered)

df['genres'] = df['genres'].apply(clean_genres)

In [41]:
df.dropna(subset = ['genres'],inplace = True)

In [43]:
df['price'] = df['price'].fillna(0)

In [45]:
df.head()

,game_name,price,review_count,developer,publisher,genres,release_year,release_month,review_sentiment,review_score
0,Slay the Spire 2,1100.0,13496,Other,Other,"Indie,Strategy,Early Access",2026,3,Overwhelmingly Positive,97
1,Resident Evil Requiem,4399.0,34291,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.","Action,Adventure",2026,2,Overwhelmingly Positive,96
2,Counter-Strike 2,0.0,2496132,Valve,Valve,"Action,Free To Play",2012,8,Very Positive,86
3,Marathon,2449.0,12119,Other,Other,Action,2026,3,Very Positive,90
4,ARC Raiders,2467.0,175346,Other,Other,Action,2025,10,Very Positive,86


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1964 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_name         1964 non-null   object 
 1   price             1964 non-null   float64
 2   review_count      1964 non-null   int32  
 3   developer         1964 non-null   object 
 4   publisher         1964 non-null   object 
 5   genres            1964 non-null   object 
 6   release_year      1964 non-null   int32  
 7   release_month     1964 non-null   int32  
 8   review_sentiment  1964 non-null   object 
 9   review_score      1964 non-null   int32  
dtypes: float64(1), int32(4), object(5)
memory usage: 138.1+ KB


In [49]:
df.isnull().sum()

game_name           0
price               0
review_count        0
developer           0
publisher           0
genres              0
release_year        0
release_month       0
review_sentiment    0
review_score        0
dtype: int64

In [53]:
df.to_csv('steam_games_cleaned.csv',index=False)

In [55]:
df2 = pd.read_csv('steam_games_cleaned.csv')

In [57]:
df2.shape

(1964, 10)

In [59]:
df2.columns

Index(['game_name', 'price', 'review_count', 'developer', 'publisher',
       'genres', 'release_year', 'release_month', 'review_sentiment',
       'review_score'],
      dtype='object')